<div align="center">
<img src="https://poorit.in/image.png" alt="Poorit" width="40" style="vertical-align: middle;"> <b>AI SYSTEMS ENGINEERING 1</b>

## Unit 4: Vector Embeddings and Databases

**CV Raman Global University, Bhubaneswar**  
*AI Center of Excellence*

---

</div>

---

### What You'll Learn

In this notebook, you will:

1. **Understand vector embeddings** - how text becomes numbers
2. **Chunk documents** for efficient retrieval
3. **Create embeddings** using sentence transformers
4. **Store vectors** in Chroma database
5. **Use API-based embeddings** with LiteLLM

---

## 1. Environment Setup

In [ ]:
# Install required packages
!pip install -q chromadb sentence-transformers scikit-learn litellm langchain-text-splitters

In [ ]:
from sentence_transformers import SentenceTransformer
import chromadb

---

## 2. What are Vector Embeddings?

Embeddings convert text into numerical vectors that capture meaning:

```
"machine learning" → [0.12, -0.45, 0.78, ...] (384 dimensions)
"AI algorithms"    → [0.11, -0.43, 0.76, ...] (similar vector)
"cooking recipes"  → [0.89, 0.23, -0.15, ...] (different vector)
```

### Key Concepts

| Concept | Description |
|---------|------------|
| **Embedding** | Dense vector representation of text |
| **Similarity** | Cosine similarity measures closeness |
| **Dimensions** | Typically 384-1536 dimensions |
| **Model** | Neural network that creates embeddings |

---

## 3. Load Embedding Model

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
texts = [
    "Machine learning is a branch of AI",
    "Deep learning uses neural networks",
    "I love cooking Indian food",
    "Cricket is popular in India"
]

embeddings = model.encode(texts)
print(f"Shape: {embeddings.shape}")  # (4, 384) → 4 texts, 384 dimensions each

---

## 4. Semantic Similarity

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(embeddings)

# Compare: similar topics vs different topics
print(f"ML vs DL (similar topics):    {similarity_matrix[0][1]:.3f}")
print(f"ML vs Cooking (different):    {similarity_matrix[0][2]:.3f}")
print(f"ML vs Cricket (different):    {similarity_matrix[0][3]:.3f}")

---
## 5. API-Based Embeddings with LiteLLM

So far we used a **local** model. Let's try **OpenAI's embedding API** through LiteLLM.

| Approach | Pros | Cons |
|----------|------|------|
| **Local** (SentenceTransformer) | Free, no API key needed | Smaller models |
| **API** (OpenAI via LiteLLM) | Higher quality models | Costs money, needs API key |

In [ ]:
import os
from getpass import getpass
from litellm import embedding

os.environ['OPENAI_API_KEY'] = getpass("Enter your OpenAI API Key: ")

In [ ]:
response = embedding(model="text-embedding-3-small", input=texts)

# See what the API returns
print(f"Model used: {response.model}")
print(f"Number of embeddings: {len(response.data)}")
print(f"OpenAI dimensions: {len(response.data[0]['embedding'])}")
print(f"Local dimensions:  {embeddings.shape[1]}")
print(f"\nFirst 5 values: {response.data[0]['embedding'][:5]}")

---

## 6. Company Knowledge Base

Let's create a knowledge base for our company. Instead of separate small documents, we'll use a single detailed text — just like real-world data.

In [ ]:
company_info = """
TechSolutions India was founded in 2018 by Priya Sharma and Rahul Verma. The company is headquartered in Bhubaneswar, Odisha, with additional offices in Bangalore and Hyderabad. TechSolutions specializes in AI/ML solutions, cloud services, and mobile applications, and has grown to over 250 employees. The company achieved ₹50 crores in annual revenue in 2024.

Priya Sharma is the CEO and co-founder. She graduated from IIT Delhi and completed her MBA from Stanford University. She previously worked as Senior Director at Infosys and won the Women in Tech Leader award in 2022. Rahul Verma is the CTO and co-founder. He graduated from BITS Pilani and previously worked as Tech Lead at Google India. His expertise is in Machine Learning and Cloud Architecture. Ananya Patel is the VP of Engineering and manages a team of 100+ engineers.

TechSolutions offers three main products. CloudAssist Pro is the flagship enterprise cloud management platform, costing ₹50,000 per month, and includes auto-scaling, real-time monitoring, cost optimization, and 24/7 support. SmartHR is an AI-powered HR management system at ₹25,000 per month that handles recruitment automation, payroll processing, performance tracking, and employee analytics. DataViz Analytics is a business intelligence platform at ₹15,000 per month with real-time dashboards, custom reports, and predictive analytics.

Work hours at TechSolutions are 9 AM to 6 PM, Monday to Friday, following a hybrid model with 3 days in office and 2 days remote. Employees receive 24 paid leaves and 10 sick leaves per year. Maternity leave is 26 weeks and paternity leave is 2 weeks. Probation period is 6 months for all new employees, with a notice period of 2 months for permanent employees and 1 month during probation.

Employee benefits include health insurance coverage of ₹5 lakh for employees and their families, covering hospitalization, OPD, dental, and vision. The learning budget is ₹50,000 per year per employee for courses, certifications, and conferences. Performance bonuses can be up to 20% of annual salary. Major clients include HDFC Bank, Tata Motors, Reliance Industries, and ICICI Bank, with a 95% customer satisfaction rate across 200+ completed projects.
"""

print(f"Document length: {len(company_info)} characters")

---

## 7. Document Chunking

For larger documents, we split them into smaller chunks.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=50)
chunks = splitter.split_text(company_info)

print(f"Split into {len(chunks)} chunks")
for i, chunk in enumerate(chunks):
    print(f"\nChunk {i+1} ({len(chunk)} chars):\n{chunk}")

---

## 8. Store in Chroma Vector Database

In [ ]:
chroma_client = chromadb.Client()

collection = chroma_client.get_or_create_collection(name="techsolutions")

print("Collection ready")

In [ ]:
chunk_embeddings = model.encode(chunks)

collection.add(
    ids=[f"chunk_{i}" for i in range(len(chunks))],
    embeddings=chunk_embeddings.tolist(),
    documents=chunks
)

print(f"Added {collection.count()} chunks to collection")

---

## 9. Semantic Search

In [ ]:
def search(query, n_results=3):
    """Search for relevant documents."""
    query_embedding = model.encode([query])
    
    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=n_results
    )
    
    return results

In [ ]:
queries = [
    "Who founded the company?",
    "What are the product prices?",
    "How many leaves do employees get?"
]

for query in queries:
    print(f"Query: {query}")
    results = search(query, n_results=2)
    for doc in results['documents'][0]:
        print(f"  → {doc[:80]}...")
    print()

---

## 10. Exercise: Expand the Knowledge Base

In [ ]:
# Exercise: Expand the knowledge base and test retrieval
# 1. Add more paragraphs to company_info (e.g., company culture, tech stack, training programs)
# 2. Re-chunk and re-index the expanded text
# 3. Test semantic search with new queries

# Example:
# company_info += """
# TechSolutions uses a modern tech stack including Python, React, and AWS...
# """

pass

---

## Key Takeaways

1. **Embeddings capture meaning** - similar texts have similar vectors

2. **Chunking is essential** - split large documents for better retrieval

3. **Vector databases enable search** - Chroma stores and queries vectors

4. **LiteLLM provides a unified API** - same embedding() call works across providers

### Popular Embedding Models

| Model | Dimensions | Use Case |
|-------|-----------|----------|
| all-MiniLM-L6-v2 | 384 | General, fast |
| all-mpnet-base-v2 | 768 | Higher quality |
| text-embedding-3-large | 3072 | OpenAI, best quality |

### What's Next?

In the next notebook, we'll build a complete RAG pipeline using LangChain.

---

## Additional Resources

- [Sentence Transformers](https://www.sbert.net/)
- [Chroma Documentation](https://docs.trychroma.com/)

---

**Course Information:**
- **Institution:** CV Raman Global University, Bhubaneswar
- **Program:** AI Center of Excellence
- **Course:** AI Systems Engineering 1
- **Developed by:** [Poorit Technologies](https://poorit.in) - *Transform Graduates into Industry-Ready Professionals*

---